In [132]:
import pandas as pd
import psycopg

In [133]:
conn = psycopg.connect(
    "dbname=dailyedge_development"
)

print("Connected")

Connected


In [134]:
parameter_sets = [
    (5, 10),
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 100),
    (75, 100),
    (75, 150),
]

In [135]:
def load_rth_session(target_date):
    rth = pd.read_sql(
        """
        SELECT timestamp, open, high, low, close, volume
        FROM CANDLES
        WHERE timestamp::date = %s
          AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
        ORDER BY timestamp
        """,
        conn,
        params=(str(target_date),)
    )

    rth["timestamp"] = pd.to_datetime(rth["timestamp"])

    return rth

In [136]:
def get_first_target(rth, opening_price, target_distance):
    upper_target = opening_price + target_distance
    lower_target = opening_price - target_distance

    target_hits = rth[
        (rth["high"] >= upper_target) |
        (rth["low"] <= lower_target)
    ]

    if target_hits.empty:
        return "Neither", None

    first_hit = target_hits.iloc[0]

    hit_upper = first_hit["high"] >= upper_target
    hit_lower = first_hit["low"] <= lower_target

    if hit_upper and hit_lower:
        return "Ambiguous", first_hit["timestamp"]

    result = "Long" if hit_upper else "Short"

    return result, first_hit["timestamp"]

In [137]:
def evaluate_trail(rth, opening_price, target_distance, trail_distance):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)
            new_stop = new_highest - trail_distance

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            # Low first
            if old_stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif new_stop_hit:
                high_first = "Failure"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if {low_first, high_first} == {"Continue", "Failure"}:
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if low_first == "Continue" and high_first == "Continue":
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + trail_distance

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            # High first
            if old_stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif new_stop_hit:
                low_first = "Failure"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if {high_first, low_first} == {"Continue", "Failure"}:
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if high_first == "Continue" and low_first == "Continue":
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [138]:
def evaluate_date(target_date, target_distance, trail_distance):
    rth = load_rth_session(target_date)

    if rth.empty:
        return None

    opening_price = rth.iloc[0]["open"]

    direction, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    trail_result = evaluate_trail(
        rth,
        opening_price,
        target_distance,
        trail_distance
    )

    return {
        "Date": target_date,
        "Direction": direction,
        "Target Time": target_time,
        "Trail Result": trail_result
    }

In [139]:
start_date = "2025-09-01"
end_date = "2026-07-07"

In [140]:
summary_rows = []

for trail_distance, target_distance in parameter_sets:
    results = []

    for date in pd.date_range(start_date, end_date):
        result = evaluate_date(
            date.date(),
            target_distance,
            trail_distance
        )

        if result is not None:
            results.append(result)

    results_df = pd.DataFrame(results)

    resolved = results_df[
        results_df["Trail Result"].isin(["Success", "Failure"])
    ]

    successes = (resolved["Trail Result"] == "Success").sum()
    failures = (resolved["Trail Result"] == "Failure").sum()
    total = len(resolved)

    summary_rows.append({
        "Trail / Target": f"{trail_distance} / {target_distance}",
        "Failure": failures,
        "Success": successes,
        "Total": total,
        "Success Rate": successes / total * 100
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider usi

,Trail / Target,Failure,Success,Total,Success Rate
0,5 / 10,0,87,87,100.000000
1,15 / 25,36,127,163,77.914110
2,25 / 50,108,97,205,47.317073
3,35 / 70,131,80,211,37.914692
4,50 / 70,92,120,212,56.603774
5,50 / 100,145,61,206,29.611650
6,75 / 100,93,113,206,54.854369
7,75 / 150,118,61,179,34.078212


In [141]:
def evaluate_breakeven_trail(
    rth,
    opening_price,
    target_distance,
    trail_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)

            # Trail upward, but never above the opening price
            new_stop = min(
                new_highest - trail_distance,
                opening_price
            )

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            # Low first
            if old_stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif new_stop_hit:
                high_first = "Failure"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if {low_first, high_first} == {"Continue", "Failure"}:
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if low_first == "Continue" and high_first == "Continue":
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)

            # Trail downward, but never below the opening price
            new_stop = max(
                new_lowest + trail_distance,
                opening_price
            )

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            # High first
            if old_stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif new_stop_hit:
                low_first = "Failure"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if {high_first, low_first} == {"Continue", "Failure"}:
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if high_first == "Continue" and low_first == "Continue":
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [142]:
breakeven_summary_rows = []

for trail_distance, target_distance in parameter_sets:
    results = []

    for date in pd.date_range(start_date, end_date):
        rth = load_rth_session(date.date())

        if rth.empty:
            continue

        opening_price = rth.iloc[0]["open"]

        trail_result = evaluate_breakeven_trail(
            rth,
            opening_price,
            target_distance,
            trail_distance
        )

        results.append(trail_result)

    resolved = [
        result for result in results
        if result in ["Success", "Failure"]
    ]

    successes = resolved.count("Success")
    failures = resolved.count("Failure")
    total = len(resolved)

    breakeven_summary_rows.append({
        "Trail / Target": f"{trail_distance} / {target_distance}",
        "Failure": failures,
        "Success": successes,
        "Total": total,
        "Success Rate": successes / total * 100
    })

breakeven_summary_df = pd.DataFrame(breakeven_summary_rows)

breakeven_summary_df

/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider usi

,Trail / Target,Failure,Success,Total,Success Rate
0,5 / 10,0,87,87,100.000000
1,15 / 25,33,131,164,79.878049
2,25 / 50,91,117,208,56.250000
3,35 / 70,107,106,213,49.765258
4,50 / 70,85,127,212,59.905660
5,50 / 100,118,89,207,42.995169
6,75 / 100,91,116,207,56.038647
7,75 / 150,103,76,179,42.458101


In [143]:
def evaluate_one_move_breakeven(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        initial_stop = opening_price - stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            # Existing stop is hit at the candle open
            if open_price <= stop:
                return "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            # Stop has already moved to breakeven
            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = high >= threshold

            # Low first: initial stop can be hit before
            # threshold/target is reached.
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                # Threshold moves stop to breakeven.
                if low <= opening_price:
                    high_first = "Failure"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            # If the threshold was reached and at least one
            # viable path survives, the stop is now breakeven.
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        initial_stop = opening_price + stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            # Existing stop is hit at the candle open
            if open_price >= stop:
                return "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            # Stop has already moved to breakeven
            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                # Threshold moves stop to breakeven.
                if high >= opening_price:
                    low_first = "Failure"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            # If the threshold was reached and at least one
            # viable path survives, the stop is now breakeven.
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [146]:
one_move_summary_rows = []

for stop_distance, target_distance in parameter_sets:
    results = []

    for date in pd.date_range(start_date, end_date):
        rth = load_rth_session(date.date())

        if rth.empty:
            continue

        opening_price = rth.iloc[0]["open"]

        result = evaluate_one_move_breakeven(
            rth,
            opening_price,
            target_distance,
            stop_distance
        )

        results.append(result)

    resolved = [
        result for result in results
        if result in ["Success", "Failure"]
    ]

    successes = resolved.count("Success")
    failures = resolved.count("Failure")
    total = len(resolved)

    one_move_summary_rows.append({
        "Stop / Target": f"{stop_distance} / {target_distance}",
        "Failure": failures,
        "Success": successes,
        "Total": total,
        "Success Rate": successes / total * 100
    })

one_move_summary_df = pd.DataFrame(one_move_summary_rows)

one_move_summary_df

/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rth = pd.read_sql(
/tmp/ipykernel_6966/2429814066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider usi

,Stop / Target,Failure,Success,Total,Success Rate
0,5 / 10,0,87,87,100.000000
1,15 / 25,25,139,164,84.756098
2,25 / 50,68,141,209,67.464115
3,35 / 70,67,145,212,68.396226
4,50 / 70,30,182,212,85.849057
5,50 / 100,79,128,207,61.835749
6,75 / 100,35,172,207,83.091787
7,75 / 150,67,112,179,62.569832
